In [ ]:
!pip install -q torch-geometric pytorch-metric-learning
!pip install -q spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 106.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import torch
import random
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F

from PIL import Image
from tqdm import tqdm

In [ ]:
from PIL import Image
from tqdm import tqdm

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoProcessor,
    CLIPVisionModel
)

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torch_geometric.data import (
    Data,
    Batch
)

from torch_geometric.nn import (
    GATv2Conv,
    global_mean_pool
)

from pytorch_metric_learning.losses import SupConLoss

from sklearn.metrics import (
    accuracy_score,
    f1_score
)

import spacy


In [ ]:

class CFG:

    TRAIN_CSV="/content/train.csv"

    DEV_CSV="/content/dev.csv"

    TEST_CSV="/content/test.csv"

    GRAPH_CACHE="/content/graph_cache.pt"

    TEST_GRAPH_CACHE="/content/test_graphs.pt"

    DEVICE="cuda" if torch.cuda.is_available() else "cpu"

    TEXT_MODEL="sentence-transformers/all-MiniLM-L6-v2"

    IMAGE_MODEL="openai/clip-vit-base-patch32"

    MAX_LEN=64

    BATCH_SIZE=16

    EPOCHS=8

    LR=2e-5

    DROPOUT=0.2

    GRAPH_DIM=256

    HIDDEN_DIM=768

    CONTRASTIVE_WEIGHT=0.2


In [ ]:
def seed_everything(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

seed_everything()

In [ ]:
nlp=spacy.load("en_core_web_sm")

In [ ]:

tokenizer=AutoTokenizer.from_pretrained(
    CFG.TEXT_MODEL
)

image_processor=AutoProcessor.from_pretrained(
    CFG.IMAGE_MODEL
)

text_encoder=AutoModel.from_pretrained(
    CFG.TEXT_MODEL
).to(CFG.DEVICE)

image_encoder=CLIPVisionModel.from_pretrained(
    CFG.IMAGE_MODEL
).to(CFG.DEVICE)

for p in text_encoder.parameters():
    p.requires_grad=False

for p in image_encoder.parameters():
    p.requires_grad=False

text_encoder.eval()
image_encoder.eval()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.layer_norm1.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias      

CLIPVisionModel(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(32, 32), stride=(32, 32), bias=False)
      (position_embedding): Embedding(50, 768)
    )
    (pre_layrnorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
        

In [ ]:

@torch.no_grad()
def build_graph(text):

    doc=nlp(text)

    tokens=[]

    for token in doc:

        if token.is_stop:
            continue

        if not token.is_alpha:
            continue

        tokens.append(
            token.text.lower()
        )

    if len(tokens)==0:
        tokens=["empty"]

    unique_tokens=list(
        dict.fromkeys(tokens)
    )

    node_map={
        tok:i for i,tok in enumerate(unique_tokens)
    }

    encoded=tokenizer(
        unique_tokens,
        padding=True,
        truncation=True,
        max_length=8,
        return_tensors="pt"
    ).to(CFG.DEVICE)

    outputs=text_encoder(
        **encoded
    )

    node_features=outputs.last_hidden_state[:,0,:].cpu()

    edges=[]

    for token in doc:

        tok=token.text.lower()
        head=token.head.text.lower()

        if tok not in node_map:
            continue

        if head not in node_map:
            continue

        src=node_map[head]
        dst=node_map[tok]

        edges.append([src,dst])

    if len(edges)==0:
        edges=[[0,0]]

    edge_index=torch.tensor(
        edges,
        dtype=torch.long
    ).t().contiguous()

    return Data(
        x=node_features,
        edge_index=edge_index
    )

def build_graph_cache(csv_path):

    df=pd.read_csv(csv_path)

    graphs=[]

    for text in tqdm(df["transcription"].astype(str).tolist()):

        graphs.append(
            build_graph(text)
        )

    return graphs


In [ ]:

if not os.path.exists(CFG.GRAPH_CACHE):

    train_graphs=build_graph_cache(
        CFG.TRAIN_CSV
    )

    dev_graphs=build_graph_cache(
        CFG.DEV_CSV
    )

    torch.save({
        "train":train_graphs,
        "dev":dev_graphs
    },CFG.GRAPH_CACHE)

graph_cache=torch.load(
    CFG.GRAPH_CACHE,
    weights_only=False
)



100%|██████████| 600/600 [00:11<00:00, 53.77it/s]


In [ ]:
class MemeDataset(Dataset):

    def __init__(self,csv_path,graphs):

        self.df=pd.read_csv(csv_path)

        self.graphs=graphs

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        image=Image.open(
            row["image_path"]
        ).convert("RGB")

        text=str(
            row["transcription"]
        )

        graph=self.graphs[idx]

        label=torch.tensor(
            row["label"],
            dtype=torch.float32
        )

        return {
            "image":image,
            "text":text,
            "graph":graph,
            "label":label
        }

def collate_fn(batch):

    images=[
        x["image"] for x in batch
    ]

    texts=[
        x["text"] for x in batch
    ]

    graphs=Batch.from_data_list(
        [x["graph"] for x in batch]
    )

    labels=torch.stack([
        x["label"] for x in batch
    ])

    return {
        "images":images,
        "texts":texts,
        "graphs":graphs,
        "labels":labels
    }


In [ ]:
class GraphEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.gat1=GATv2Conv(
            384,
            CFG.GRAPH_DIM,
            heads=4,
            dropout=CFG.DROPOUT
        )

        self.gat2=GATv2Conv(
            CFG.GRAPH_DIM*4,
            CFG.GRAPH_DIM,
            heads=2,
            dropout=CFG.DROPOUT
        )

        self.gat3=GATv2Conv(
            CFG.GRAPH_DIM*2,
            CFG.GRAPH_DIM,
            heads=1,
            concat=False,
            dropout=CFG.DROPOUT
        )

        self.proj=nn.Sequential(

            nn.Linear(
                CFG.GRAPH_DIM,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU()
        )

    def forward(self,graph):

        x=graph.x.to(CFG.DEVICE)

        edge_index=graph.edge_index.to(
            CFG.DEVICE
        )

        batch=graph.batch.to(
            CFG.DEVICE
        )

        x=self.gat1(
            x,
            edge_index
        )

        x=F.elu(x)

        x=self.gat2(
            x,
            edge_index
        )

        x=F.elu(x)

        x=self.gat3(
            x,
            edge_index
        )

        x=global_mean_pool(
            x,
            batch
        )

        x=self.proj(x)

        return x


In [ ]:

class ContradictionFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder=nn.Sequential(

            nn.Linear(
                384*2,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU(),

            nn.Dropout(CFG.DROPOUT)
        )

    def forward(
        self,
        text_emb,
        image_emb
    ):

        diff=torch.abs(
            text_emb-image_emb
        )

        mul=text_emb*image_emb

        x=torch.cat([
            diff,
            mul
        ],dim=1)

        return self.encoder(x)


In [ ]:

class MFBFusion(nn.Module):

    def __init__(self):

        super().__init__()

        self.text_proj=nn.Linear(
            384,
            CFG.HIDDEN_DIM
        )

        self.image_proj=nn.Linear(
            384,
            CFG.HIDDEN_DIM
        )

        self.out_proj=nn.Sequential(

            nn.Linear(
                CFG.HIDDEN_DIM,
                384
            ),

            nn.LayerNorm(384),

            nn.GELU()
        )

    def forward(
        self,
        text_emb,
        image_emb
    ):

        t=self.text_proj(
            text_emb
        )

        i=self.image_proj(
            image_emb
        )

        fused=t*i

        fused=self.out_proj(
            fused
        )

        return fused


In [ ]:

# class DynamicGating(nn.Module):

#     def __init__(self):

#         super().__init__()

#         self.gate_text=nn.Linear(
#             384,
#             1
#         )

#         self.gate_image=nn.Linear(
#             384,
#             1
#         )

#         self.gate_graph=nn.Linear(
#             384,
#             1
#         )

#         self.gate_contra=nn.Linear(
#             384,
#             1
#         )

#     def forward(
#         self,
#         text_emb,
#         image_emb,
#         graph_emb,
#         contra_emb
#     ):

#         wt=self.gate_text(text_emb)

#         wi=self.gate_image(image_emb)

#         wg=self.gate_graph(graph_emb)

#         wc=self.gate_contra(contra_emb)

#         weights=torch.cat([
#             wt,
#             wi,
#             wg,
#             wc
#         ],dim=1)

#         weights=torch.softmax(
#             weights,
#             dim=1
#         )

#         wt=weights[:,0:1]
#         wi=weights[:,1:2]
#         wg=weights[:,2:3]
#         wc=weights[:,3:4]

#         fused=(
#             wt*text_emb+
#             wi*image_emb+
#             wg*graph_emb+
#             wc*contra_emb
#         )

#         return fused


In [ ]:
# class InterMemeReasoning(nn.Module):

#     def __init__(self):

#         super().__init__()

#         self.phi=nn.Linear(
#             384,
#             384
#         )

#         self.gamma=nn.Linear(
#             384,
#             384
#         )

#         self.g=nn.Linear(
#             384,
#             384
#         )

#         self.r=nn.Linear(
#             384,
#             384
#         )

#         self.norm=nn.LayerNorm(
#             384
#         )

#     def forward(self,x):

#         q=self.phi(x)

#         k=self.gamma(x)

#         affinity=torch.matmul(
#             q,
#             k.transpose(0,1)
#         )

#         affinity=affinity / x.size(0)

#         msg=torch.matmul(
#             affinity,
#             self.g(x)
#         )

#         out=self.r(msg)

#         out=out+x

#         out=self.norm(out)

#         return out



In [ ]:
class MemeModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.graph_encoder=GraphEncoder()

        self.contradiction=ContradictionFusion()

        self.mfb=MFBFusion()

        # self.gating=DynamicGating()

        # self.inter_meme=InterMemeReasoning()

        self.image_proj=nn.Linear(
            768,
            384
        )

        self.final_fusion=nn.Sequential(

            nn.Linear(
                384*5,
                768
            ),

            nn.LayerNorm(768),

            nn.GELU(),

            nn.Dropout(CFG.DROPOUT),

            nn.Linear(
                768,
                384
            )
        )

        self.projection_head=nn.Sequential(

            nn.Linear(
                384,
                256
            ),

            nn.GELU(),

            nn.Linear(
                256,
                128
            )
        )

        self.classifier=nn.Sequential(

            nn.Linear(
                384,
                256
            ),

            nn.LayerNorm(256),

            nn.GELU(),

            nn.Dropout(CFG.DROPOUT),

            nn.Linear(
                256,
                1
            )
        )

    @torch.no_grad()
    def encode_text(self,texts):

        encoded=tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=CFG.MAX_LEN,
            return_tensors="pt"
        ).to(CFG.DEVICE)

        outputs=text_encoder(
            **encoded
        )

        return outputs.last_hidden_state[:,0,:]

    @torch.no_grad()
    def encode_image(self,images):

        encoded=image_processor(
            images=images,
            return_tensors="pt"
        ).to(CFG.DEVICE)

        outputs=image_encoder(
            **encoded
        )

        emb=outputs.pooler_output

        emb=self.image_proj(
            emb
        )

        return emb

    def forward(self,batch):

        text_emb=self.encode_text(
            batch["texts"]
        )

        image_emb=self.encode_image(
            batch["images"]
        )

        graph_emb=self.graph_encoder(
            batch["graphs"]
        )

        contra_emb=self.contradiction(
            text_emb,
            image_emb
        )

        multimodal_emb=self.mfb(
            text_emb,
            image_emb
        )

        # gated_emb=self.gating(
        #     text_emb,
        #     image_emb,
        #     graph_emb,
        #     contra_emb
        # )

        fusion=torch.cat([
            text_emb,
            image_emb,
            graph_emb,
            contra_emb,
            multimodal_emb
        ],dim=1)

        fusion=self.final_fusion(
            fusion
        )

        # fusion=fusion+gated_emb

        fusion=self.inter_meme(
            fusion
        )

        logits=self.classifier(
            fusion
        ).squeeze(1)

        proj=F.normalize(
            self.projection_head(
                fusion
            ),
            dim=1
        )

        return logits,proj


In [ ]:
train_dataset=MemeDataset(
    CFG.TRAIN_CSV,
    graph_cache["train"]
)

dev_dataset=MemeDataset(
    CFG.DEV_CSV,
    graph_cache["dev"]
)

train_loader=DataLoader(
    train_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=True
)

dev_loader=DataLoader(
    dev_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=True
)


In [ ]:
model=MemeModel().to(
    CFG.DEVICE
)

bce_loss=nn.BCEWithLogitsLoss()

contrastive_loss=SupConLoss()

optimizer=torch.optim.AdamW(
    model.parameters(),
    lr=CFG.LR,
    weight_decay=1e-4
)

scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG.EPOCHS
)

best_f1=0
best_threshold=0.5



In [ ]:
for epoch in range(CFG.EPOCHS):

    model.train()

    losses=[]

    for batch in tqdm(train_loader):

        labels=batch["labels"].to(
            CFG.DEVICE
        )

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            logits,proj=model(batch)

            cls_loss=bce_loss(
                logits,
                labels
            )

            cont_loss=contrastive_loss(
                proj,
                labels.long()
            )

            loss=(
                cls_loss+
                CFG.CONTRASTIVE_WEIGHT*
                cont_loss
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        losses.append(
            loss.item()
        )

    scheduler.step()

    print(
        f"Epoch {epoch+1} Loss:",
        np.mean(losses)
    )

    # ========================================================
    # EVAL
    # ========================================================

    model.eval()

    all_probs=[]
    all_labels=[]

    with torch.no_grad():

        for batch in tqdm(dev_loader):

            labels=batch["labels"].to(
                CFG.DEVICE
            )

            with torch.amp.autocast("cuda"):

                logits,_=model(batch)

            probs=torch.sigmoid(
                logits
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

    all_probs=np.array(all_probs)

    all_labels=np.array(all_labels)

    best_epoch_f1=0
    best_epoch_thr=0.5

    for thr in np.arange(
        0.1,
        0.9,
        0.01
    ):

        preds=(
            all_probs>thr
        ).astype(int)

        score=f1_score(
            all_labels,
            preds,
            average="macro"
        )

        if score>best_epoch_f1:

            best_epoch_f1=score

            best_epoch_thr=thr

    preds=(
        all_probs>best_epoch_thr
    ).astype(int)

    acc=accuracy_score(
        all_labels,
        preds
    )

    print(
        "Accuracy:",
        acc
    )

    print(
        "Macro F1:",
        best_epoch_f1
    )

    print(
        "Threshold:",
        best_epoch_thr
    )

    if best_epoch_f1>best_f1:

        best_f1=best_epoch_f1

        best_threshold=best_epoch_thr

        torch.save(
            model.state_dict(),
            "best_model.pt"
        )

        # print(
        #     "Best Model Saved"
        # )



model.load_state_dict(
    torch.load(
        "best_model.pt"
    )
)

model.eval()



In [ ]:
if not os.path.exists(
    CFG.TEST_GRAPH_CACHE
):

    test_graphs=build_graph_cache(
        CFG.TEST_CSV
    )

    torch.save(
        test_graphs,
        CFG.TEST_GRAPH_CACHE
    )

test_graphs=torch.load(
    CFG.TEST_GRAPH_CACHE,
    weights_only=False
)

# ============================================================
# TEST DATASET
# ============================================================

class TestDataset(Dataset):

    def __init__(self,csv_path,graphs):

        self.df=pd.read_csv(csv_path)

        self.graphs=graphs

    def __len__(self):

        return len(self.df)

    def __getitem__(self,idx):

        row=self.df.iloc[idx]

        image=Image.open(
            row["image_path"]
        ).convert("RGB")

        text=str(
            row["transcription"]
        )

        graph=self.graphs[idx]

        return {
            "image":image,
            "text":text,
            "graph":graph
        }

def test_collate_fn(batch):

    images=[
        x["image"] for x in batch
    ]

    texts=[
        x["text"] for x in batch
    ]

    graphs=Batch.from_data_list(
        [x["graph"] for x in batch]
    )

    return {
        "images":images,
        "texts":texts,
        "graphs":graphs
    }

test_dataset=TestDataset(
    CFG.TEST_CSV,
    test_graphs
)

test_loader=DataLoader(
    test_dataset,
    batch_size=CFG.BATCH_SIZE,
    shuffle=False,
    collate_fn=test_collate_fn
)

# ============================================================
# TEST INFERENCE
# ============================================================



# ============================================================
# SAVE
# ============================================================

100%|██████████| 1000/1000 [00:15<00:00, 66.57it/s]


In [ ]:
all_probs=[]
all_preds=[]

with torch.no_grad():

    for batch in tqdm(test_loader):

        with torch.amp.autocast("cuda"):

            logits,_=model(batch)

        probs=torch.sigmoid(
            logits
        )

        preds=(
            probs>best_threshold
        ).long()

        all_probs.extend(
            probs.cpu().numpy()
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

100%|██████████| 63/63 [00:17<00:00,  3.62it/s]


In [ ]:
# ============================================================
# SAVE
# ============================================================

test_df=pd.read_csv(
    CFG.TEST_CSV
)

test_df["prediction"]=all_preds

test_df["probability"]=all_probs

test_df.to_csv(
    "final_predictions.csv",
    index=False
)

print(
    "Predictions Saved"
)

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

pred_df = pd.read_csv("/content/final_predictions.csv")
true_df = pd.read_csv("/content/test.csv")

true_df["indian_labels"] = true_df["indian_labels"].map({
    "not-misogyny": 0,
    "misogyny": 1
})

merged = pred_df.merge(true_df, on="image_id")
y_pred = merged["prediction"]
y_true = merged["indian_labels"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")
print("Accuracy :", acc)
print("Macro F1 :", macro_f1)

Accuracy : 0.7448
Macro F1 : 0.7301
